## Load RUCA CSV and drop secondary RUCA columns

This notebook loads `RUCA-codes-2020-tract.csv` into a pandas DataFrame and removes the secondary RUCA fields:

- `SecondaryRUCA`
- `SecondaryRUCADescription`
- `SecondaryDestinationCode`
- `SecondaryDestinationName`


In [ ]:
# If you get an import error for pandas (or Polars isn't installed), run ONE of these:
#
# !python -m pip install -U "numpy" "pandas"
# !python -m pip install -U "polars"


In [4]:
from pathlib import Path

# Prefer pandas, but fall back to Polars if pandas can't import in this environment.
try:
    import pandas as pd  # type: ignore
    DF_LIB = "pandas"
except Exception as e:  # noqa: BLE001
    pd = None  # type: ignore
    DF_LIB = "polars"
    import polars as pl  # type: ignore

DF_LIB


'pandas'

In [5]:
csv_path = Path("../RUCA-codes-2020-tract.csv")
csv_path.exists(), csv_path.resolve()


(True,
 PosixPath('/Users/shashin/Documents/GitHub/INT Biohackathon 2026/RUCA-codes-2020-tract.csv'))

In [6]:
# Read as strings to preserve leading zeros in FIPS-like columns.
# This CSV is not guaranteed to be UTF-8; try a couple common encodings.
if DF_LIB == "pandas":
    try:
        df_raw = pd.read_csv(csv_path, dtype=str, encoding="utf-8")  # type: ignore[union-attr]
    except UnicodeDecodeError:
        # Latin-1 / cp1252 will successfully decode any byte sequence.
        df_raw = pd.read_csv(csv_path, dtype=str, encoding="latin1")  # type: ignore[union-attr]
else:
    try:
        df_raw = pl.read_csv(csv_path, dtypes=pl.Utf8, encoding="utf8")  # type: ignore[name-defined]
    except Exception:
        df_raw = pl.read_csv(csv_path, dtypes=pl.Utf8, encoding="latin1")  # type: ignore[name-defined]

df_raw.shape, (df_raw.columns.tolist()[:10] if DF_LIB == "pandas" else df_raw.columns[:10])


((85528, 27),
 ['TractFIPS23',
  'CountyFIPS23',
  'CountyCode23',
  'CountyName23',
  'TractFIPS20',
  'TractCode20',
  'TractName20',
  'CountyFIPS20',
  'CountyCode20',
  'CountyName20'])

In [13]:
SECONDARY_RUCA_COLS = [
    "SecondaryRUCA",
    "SecondaryRUCADescription",
    "SecondaryDestinationCode",
    "SecondaryDestinationName",
]

DROP_COLS = [
    "TractFIPS23",
    "CountyFIPS23",
    "CountyCode23",
    "TractFIPS20",
    "TractCode20",
    "TractName20",
    "CountyFIPS20",
    "CountyCode20",
]

if DF_LIB == "pandas":
    df = df_raw.drop(columns=SECONDARY_RUCA_COLS, errors="ignore").copy()  # type: ignore[union-attr]

    # Filter out PrimaryRUCA <= 3 (keep > 3)
    primary_ruca_num = pd.to_numeric(df["PrimaryRUCA"], errors="coerce")  # type: ignore[union-attr]
    df = df.loc[primary_ruca_num > 3].copy()

    # Drop requested columns
    df = df.drop(columns=DROP_COLS, errors="ignore")
else:
    df = df_raw.drop([c for c in SECONDARY_RUCA_COLS if c in df_raw.columns])

    # Filter out PrimaryRUCA <= 3 (keep > 3)
    df = df.with_columns(
        pl.col("PrimaryRUCA").cast(pl.Int64, strict=False).alias("PrimaryRUCA__num")
    )
    df = df.filter(pl.col("PrimaryRUCA__num") > 3).drop("PrimaryRUCA__num")

    # Drop requested columns
    df = df.drop([c for c in DROP_COLS if c in df.columns])

df.shape


(17050, 15)

In [14]:
raw_cols = (df_raw.columns.tolist() if DF_LIB == "pandas" else df_raw.columns)
missing = [c for c in SECONDARY_RUCA_COLS if c not in raw_cols]
dropped = [c for c in SECONDARY_RUCA_COLS if c in raw_cols]

{
    "dropped_secondary_ruca_columns": dropped,
    "secondary_ruca_columns_missing_from_csv": missing,
}


{'dropped_secondary_ruca_columns': ['SecondaryRUCA',
  'SecondaryRUCADescription',
  'SecondaryDestinationCode',
  'SecondaryDestinationName'],
 'secondary_ruca_columns_missing_from_csv': []}

In [15]:
df.head()


,CountyName23,CountyName20,StateFIPS20,StateName20,UrbanAreaCode20,UrbanAreaName20,UrbanCore,UrbanCoreType,PrimaryRUCA,PrimaryRUCADescription,PrimaryDestinationCode,PrimaryDestinationName,Population,LandArea,PopDensity
17,Baldwin County,Baldwin County,01,Alabama,NaN,NaN,0,Rural,8,Small town high commuting,5923,"Bay Minette, AL",3745,375.1,10
18,Baldwin County,Baldwin County,01,Alabama,NaN,NaN,0,Rural,9,Small town low commuting,5923,"Bay Minette, AL",3000,85.6,35
21,Baldwin County,Baldwin County,01,Alabama,05923,"Bay Minette, AL",1,Small town core,7,Small town core,5923,"Bay Minette, AL",4850,6.7,720.9
22,Baldwin County,Baldwin County,01,Alabama,05923,"Bay Minette, AL",1,Small town core,7,Small town core,5923,"Bay Minette, AL",3234,7.2,450.9
32,Baldwin County,Baldwin County,01,Alabama,NaN,NaN,0,Rural,10,Rural area,1003010904,Census Tract 109.04; Baldwin County; Alabama,7747,106.6,72.7


### (Optional) Save a cleaned copy

Uncomment to write a new CSV without the secondary RUCA columns.


In [ ]:
# out_path = Path("../RUCA-codes-2020-tract.cleaned.csv")
# if DF_LIB == "pandas":
#     df.to_csv(out_path, index=False)
# else:
#     df.write_csv(out_path)
# out_path
